# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Sir Richard Bishop, Artur Rumiński w Alchemii
2. Kino Bambino. Milość, która zostaje
3. Granica kobiecości
4. Femme fatale
5. Piosenka jest dobra na wszystko
6. Paw królowej
7. Burza
8. Wstyd (Teatr Ludowy)
9. Wariacje Tischnerowskie. Kabaret filozoficzny
10. Wszystkie sonaty fortepianowe Beethovena: The Late Beethoven

Czas wykonania: 3.34s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Sir Richard Bishop, Artur Rumiński w Alchemii
2. Kino Bambino. Milość, która zostaje
3. Granica kobiecości
4. Femme fatale
5. Piosenka jest dobra na wszystko
6. Paw królowej
7. Burza
8. Wstyd (Teatr Ludowy)
9. Wariacje Tischnerowskie. Kabaret filozoficzny
10. Wszystkie sonaty fortepianowe Beethovena: The Late Beethoven

Czas wykonania (wątki): 0.92s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [4]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.64s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [5]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def fetch_cat_fact(dummy=None):
    url = "https://catfact.ninja/fact"
    fakt = requests.get(url).json().get("fact")

    return fakt

def multithreading():
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=6) as executor:
        wyniki = executor.map(fetch_cat_fact, [None]*20)
        diff = end - start
        
    for fakt in wyniki:
        print(fakt)

    end = time.time()
    

    print(diff)

def sequence():
    start = time.time()

    for i in range(20):
        print(fetch_cat_fact())

    end = time.time()
    diff = end - start
    print(diff)

def main():
    sequence()

if __name__ == "__main__":
    main()

According to Hebrew legend, Noah prayed to God for help protecting all the food he stored on the ark from being eaten by rats. In reply, God made the lion sneeze, and out popped a cat.
Mountain lions are strong jumpers, thanks to muscular hind legs that are longer than their front legs.
Cats have about 130,000 hairs per square inch (20,155 hairs per square centimeter).
The ancestor of all domestic cats is the African Wild Cat which still exists today.
The richest cat is Blackie who was left £15 million by his owner, Ben Rea.
About 37% of American homes today have at least 1 cat.
When a cat drinks, its tongue - which has tiny barbs on it - scoops the liquid up backwards.
Cats have the largest eyes of any mammal.
The first cat in space was a French cat named Felicette (a.k.a. “Astrocat”) In 1963, France blasted the cat into outer space. Electrodes implanted in her brains sent neurological signals back to Earth. She survived the trip.
Neutering a male cat will, in almost all cases, stop h

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [6]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

def producent(q_parzyste, q_nieparzyste):
    for i in range(1, 101):
        if i % 2 == 0:
            q_parzyste.put(i)
        else:
            q_nieparzyste.put(i)
    pass
    print("Producent zakończył generowanie liczb.")

def consument(que, consument_name):
    while True:
        liczba = que.get()
        if liczba == None:
            break
        
        print(f"{consument_name} pobrał liczbę {liczba}")
        que.task_done()
        if liczba == 100:
            break

def main():
    czas = time.time()
    q_parzyste = queue.Queue()
    q_nieparzyste = queue.Queue()
    
    producent_thread = threading.Thread(target=producent, args=(q_parzyste, q_nieparzyste))

    consument1_thread = threading.Thread(target=consument, args=(q_parzyste, "Konsument parzysty"))    
    consument2_thread = threading.Thread(target=consument, args=(q_nieparzyste, "Konsument Nieparzysty"))    

    producent_thread.start()
    consument1_thread.start()
    consument2_thread.start()
    
    producent_thread.join()
    q_parzyste.put(None)
    q_nieparzyste.put(None)
    consument1_thread.join()
    consument2_thread.join()

    print(time.time() - czas)


if __name__ == "__main__":
    main()

Producent zakończył generowanie liczb.Konsument parzysty pobrał liczbę 2
Konsument parzysty pobrał liczbę 4
Konsument parzysty pobrał liczbę 6
Konsument parzysty pobrał liczbę 8
Konsument parzysty pobrał liczbę 10
Konsument parzysty pobrał liczbę 12
Konsument parzysty pobrał liczbę 14
Konsument parzysty pobrał liczbę 16
Konsument parzysty pobrał liczbę 18
Konsument parzysty pobrał liczbę 20
Konsument parzysty pobrał liczbę 22
Konsument parzysty pobrał liczbę 24
Konsument parzysty pobrał liczbę 26
Konsument parzysty pobrał liczbę 28
Konsument parzysty pobrał liczbę 30
Konsument parzysty pobrał liczbę 32
Konsument parzysty pobrał liczbę 34
Konsument parzysty pobrał liczbę 36
Konsument parzysty pobrał liczbę 38
Konsument parzysty pobrał liczbę 40
Konsument parzysty pobrał liczbę 42
Konsument parzysty pobrał liczbę 44
Konsument parzysty pobrał liczbę 46
Konsument parzysty pobrał liczbę 48
Konsument parzysty pobrał liczbę 50
Konsument parzysty pobrał liczbę 52
Konsument parzysty pobrał licz

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [ ]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    # Twój kod tutaj...
    pass